# Quant Crowding Detector

**What This Project Is ?**

This project builds a real-time crowding risk monitor for quantitative hedge funds using entirely public data. Crowding happens when multiple quant funds converge on the same stock positions simultaneously. When one fund is forced to sell, the others follow, triggering a cascade of losses across the entire quant ecosystem. This is exactly what happened in August 2007 and again in summer 2025.

**Goal**

The goal is to detect which stocks are most at risk of a crowding-driven unwind before it happens, using SEC regulatory filings, market volume data, academic factor research, and machine learning anomaly detection — all combined into a single interactive dashboard with an AI analyst layer

**Datasets Used**

The first dataset is SEC EDGAR 13F filings pulled directly from the US Securities and Exchange Commission. This gives us the exact stock holdings of 8 major quant funds including Two Sigma, Renaissance Technologies, DE Shaw, AQR Capital Management, Bridgewater Associates, Jane Street, Point72 and Goldman Sachs Asset Management.

The second dataset is daily trading volume pulled from yfinance for the top 200 stocks by hedge fund ownership. This is used to calculate Days ADV — the number of days it would take all funds to exit a position under normal market conditions.

The third dataset is the Ken French Five Factor daily returns library covering 2020 to 2025. This tracks how momentum, value, size and profitability factors performed over time and allows us to mark historical quant unwind events on the timeline.

The fourth dataset is a corpus of 15 academic research papers on quant crowding, factor investing and systemic risk downloaded from arXiv. These papers power the RAG knowledge base that the AI analyst draws from when answering questions.

**What the Dashboard Does ?**

The dashboard is built in Streamlit and served publicly via ngrok. It shows a bar chart of the top 20 most crowded stocks ranked by crowding score, a scatter plot risk map showing exit difficulty versus fund concentration, a table of the 17 stocks flagged as anomalously crowded by the Isolation Forest model, a factor performance chart with major unwind events marked, and a treemap showing crowded positions sized by total dollar value at risk.
At the bottom of the dashboard an AI analyst automatically generates a Monday morning style risk brief based on the live data every time the app loads. Users can also ask any question about crowding risk and the AI answers using both the live SEC data and citations from the 15 research papers simultaneously.


In [ ]:
!pip install yfinance pandas numpy scikit-learn matplotlib seaborn plotly -q
!pip install langchain langchain-community chromadb sentence-transformers -q
!pip install langchain-groq pypdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import plotly.express as px


**Dataset 1 — SEC 13F Holdings**

Definining verified funds

I have chosen to work with these 8 funds because in quantitative finance, it is not about how many funds you track — it is about which ones you track. These 8 firms, Two Sigma, Renaissance Technologies, AQR, DE Shaw, Point72, Bridgewater, Jane Street and Goldman Sachs Asset Management, collectively manage trillions of dollars using systematic strategies. If a crowding event is going to happen in the market, it will originate from portfolios exactly like these. Tracking 8 of the right funds tells you more than tracking 50 of the wrong ones.

In [ ]:
import requests
import pandas as pd
import time

headers = {"User-Agent": "nayanibhatta@gmail.com"}  # replace with your email

# 8 verified quant funds — CIKs confirmed correct
funds = {
    "Two Sigma Investments":    "0001179392",
    "Renaissance Technologies": "0001037389",
    "DE Shaw":                  "0001009207",
    "AQR Capital Management":   "0001167557",
    "Point72":                  "0001603466",
    "Bridgewater Associates":   "0001350694",
    "Jane Street":              "0001595888",
    "Goldman Sachs Asset Mgmt": "0000886982",
}

**Dataset 1 - Get latest 13F filing info for each fund**



In [ ]:
def get_latest_13f(fund_name, cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    try:
        r = requests.get(url, headers=headers, timeout=10)
        data = r.json()

        filings = data["filings"]["recent"]
        forms   = filings["form"]
        dates   = filings["filingDate"]
        accessions = filings["accessionNumber"]

        for i, form in enumerate(forms):
            if form == "13F-HR":
                return {
                    "fund":      fund_name,
                    "cik":       cik,
                    "date":      dates[i],
                    "accession": accessions[i].replace("-", "")
                }
    except Exception as e:
        print(f" {fund_name} — {e}")

    return None

# Pull latest 13F for all 8 funds
filings = []
for name, cik in funds.items():
    result = get_latest_13f(name, cik)
    if result:
        filings.append(result)
        print(f" {name:35s} — latest 13F: {result['date']}")
    else:
        print(f" {name:35s} — no 13F found")
    time.sleep(0.3)

filings_df = pd.DataFrame(filings)
print(f"\n{len(filings_df)} funds ready")
print(filings_df[["fund", "date"]])

 Two Sigma Investments               — latest 13F: 2026-02-17
 Renaissance Technologies            — latest 13F: 2026-02-12
 DE Shaw                             — latest 13F: 2026-02-17
 AQR Capital Management              — latest 13F: 2026-02-17
 Point72                             — latest 13F: 2026-02-17
 Bridgewater Associates              — latest 13F: 2026-02-13
 Jane Street                         — latest 13F: 2026-02-12
 Goldman Sachs Asset Mgmt            — latest 13F: 2026-02-10

8 funds ready
                       fund        date
0     Two Sigma Investments  2026-02-17
1  Renaissance Technologies  2026-02-12
2                   DE Shaw  2026-02-17
3    AQR Capital Management  2026-02-17
4                   Point72  2026-02-17
5    Bridgewater Associates  2026-02-13
6               Jane Street  2026-02-12
7  Goldman Sachs Asset Mgmt  2026-02-10


Pull actual stock holdings for each fund

In [ ]:
import time

def get_holdings_fixed(fund_name, cik, accession):
    index_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/index.json"

    try:
        r = requests.get(index_url, headers=headers, timeout=30)
        files = r.json().get("directory", {}).get("item", [])

        xml_file = None
        for f in files:
            if f["name"].endswith(".xml") and "primary_doc" not in f["name"].lower():
                xml_file = f["name"]
                break

        if not xml_file:
            print(f" {fund_name} — no XML found")
            return None

        xml_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{xml_file}"
        xml_r = requests.get(xml_url, headers=headers, timeout=30)

        from xml.etree import ElementTree as ET
        root = ET.fromstring(xml_r.content)

        holdings = []
        for info in root.iter():
            if "infoTable" in info.tag:
                holding = {}
                for child in info:
                    tag = child.tag.split("}")[-1]

                    # THIS is the fix — handles nested shares tag
                    if tag == "shrsOrPrnAmt":
                        for subchild in child:
                            subtag = subchild.tag.split("}")[-1]
                            holding[subtag] = subchild.text
                    else:
                        holding[tag] = child.text

                holdings.append(holding)

        if holdings:
            df = pd.DataFrame(holdings)
            df["fund"] = fund_name
            return df

    except Exception as e:
        print(f" {fund_name} — {e}")
        return None

# Re-pull all funds with fixed parser
all_holdings_fixed = []

for _, row in filings_df.iterrows():
    print(f"Pulling: {row['fund']}...")
    df = get_holdings_fixed(row["fund"], row["cik"], row["accession"])
    if df is not None:
        all_holdings_fixed.append(df)
        print(f"   {len(df)} positions")
    time.sleep(5)

# Combine
raw_fixed_df = pd.concat(all_holdings_fixed, ignore_index=True)
print(f"\n Total rows: {len(raw_fixed_df)}")
print(f"\nShares sample:")
print(raw_fixed_df["sshPrnamt"].head(10).tolist())


Pulling: Two Sigma Investments...
   4041 positions
Pulling: Renaissance Technologies...
   3185 positions
Pulling: DE Shaw...
   5839 positions
Pulling: AQR Capital Management...
   16934 positions
Pulling: Point72...
   3862 positions
Pulling: Bridgewater Associates...
   1040 positions
Pulling: Jane Street...
   17120 positions
Pulling: Goldman Sachs Asset Mgmt...
   13034 positions

 Total rows: 65055

Shares sample:
['164595', '1155691', '150624', '47902', '353330', '2511590', '13627', '14512', '6000', '2600']


Cleaning the data

In [ ]:
# Clean the fixed dataset
holdings_df = raw_fixed_df.copy()

# Rename columns clearly
holdings_df = holdings_df.rename(columns={
    "nameOfIssuer": "stock",
    "sshPrnamt":    "shares",
    "value":        "value_usd"
})

# Convert to numeric
holdings_df["value_usd"] = pd.to_numeric(holdings_df["value_usd"], errors="coerce")
holdings_df["shares"]    = pd.to_numeric(holdings_df["shares"], errors="coerce")

# Keep only what we need
holdings_df = holdings_df[["fund", "stock", "cusip", "value_usd", "shares"]]

# Drop missing values
holdings_df = holdings_df.dropna(subset=["value_usd", "cusip", "stock", "shares"])

# Drop duplicates
holdings_df = holdings_df.drop_duplicates(subset=["fund", "cusip"])


In [ ]:
# Sanity check
print(f"Clean rows: {len(holdings_df)}")
print(f"Funds: {holdings_df['fund'].nunique()}")
print(f"Unique stocks: {holdings_df['cusip'].nunique()}")
print(f"Total value: ${holdings_df['value_usd'].sum()/1e9:.1f} billion")
print(f"Shares NaN count: {holdings_df['shares'].isna().sum()}")
print(f"\nTop 10 most held stocks:")
print(
    holdings_df.groupby("stock")["value_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .apply(lambda x: f"${x/1e6:.0f}M")
)

Clean rows: 29061
Funds: 8
Unique stocks: 8534
Total value: $394.0 billion
Shares NaN count: 0

Top 10 most held stocks:
stock
ISHARES TR                   $11869M
NVIDIA CORPORATION            $9976M
SPDR S&P 500 ETF TR           $9365M
SPDR SERIES TRUST             $5947M
GOLDMAN SACHS ETF TR          $3885M
AMAZON COM INC                $3313M
PALANTIR TECHNOLOGIES INC     $3291M
MICROSOFT CORP                $2870M
ALPHABET INC                  $2859M
ISHARES INC                   $2596M
Name: value_usd, dtype: object


**Dataset 2 — Daily Trading Volume from yfinance**

Pull Daily Volume from yfinance

In [ ]:
import yfinance as yf
import numpy as np

# Get unique stocks from our holdings
# yfinance needs ticker symbols not CUSIP
# We will use stock names to get tickers — focus on top 500 by value
top_stocks = (
    holdings_df.groupby("stock")["value_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(500)
    .index.tolist()
)

print(f"Top 500 stocks by hedge fund value")
print(f"Sample: {top_stocks[:10]}")

Top 500 stocks by hedge fund value
Sample: ['ISHARES TR', 'NVIDIA CORPORATION', 'SPDR S&P 500 ETF TR', 'SPDR SERIES TRUST', 'GOLDMAN SACHS ETF TR', 'AMAZON COM INC', 'PALANTIR TECHNOLOGIES INC', 'MICROSOFT CORP', 'ALPHABET INC', 'ISHARES INC']


Pulling the daily trading volume for all 188 tickers from yfinance

In [ ]:
def cusip_to_ticker(cusips):
    """
    Converts CUSIP numbers to ticker symbols using OpenFIGI API
    Free, no API key needed for basic usage
    """
    url = "https://api.openfigi.com/v3/mapping"
    headers_figi = {"Content-Type": "application/json"}

    tickers = {}

    # Process in batches of 10 (API limit)
    batch_size = 10
    cusip_list = list(cusips)

    for i in range(0, min(len(cusip_list), 500), batch_size):
        batch = cusip_list[i:i+batch_size]
        payload = [{"idType": "ID_CUSIP", "idValue": c} for c in batch]

        try:
            r = requests.post(url, json=payload, headers=headers_figi, timeout=10)
            results = r.json()

            for cusip, result in zip(batch, results):
                if result and "data" in result and result["data"]:
                    ticker = result["data"][0].get("ticker", None)
                    if ticker:
                        tickers[cusip] = ticker

        except Exception as e:
            print(f"Batch {i} failed — {e}")

        time.sleep(0.5)

        if i % 50 == 0:
            print(f"Processed {i}/{min(len(cusip_list), 500)} CUSIPs...")

    return tickers

# Get top 200 CUSIPs by total value across all funds
top_cusips = (
    holdings_df.groupby("cusip")["value_usd"]
    .sum()
    .sort_values(ascending=False)
    .head(200)
    .index.tolist()
)

print(f"Converting top 200 CUSIPs to tickers...")
cusip_ticker_map = cusip_to_ticker(top_cusips)
print(f"\n Successfully mapped {len(cusip_ticker_map)} CUSIPs to tickers")
print(f"\nSample mapping:")
for cusip, ticker in list(cusip_ticker_map.items())[:10]:
    print(f"  {cusip} → {ticker}")

Converting top 200 CUSIPs to tickers...
Processed 0/200 CUSIPs...
Processed 50/200 CUSIPs...
Processed 100/200 CUSIPs...
Processed 150/200 CUSIPs...

 Successfully mapped 188 CUSIPs to tickers

Sample mapping:
  67066G104 → NVDA
  78462F103 → SPY
  464287200 → IVV
  023135106 → AMZN
  69608A108 → PLTR
  594918104 → MSFT
  874039100 → TSM
  595112103 → MU
  88160R101 → TSLA
  110122108 → BMY


In [ ]:
tickers = list(cusip_ticker_map.values())

print(f"Pulling volume data for {len(tickers)} stocks...")

volume_data = {}

for i, ticker in enumerate(tickers):
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="3mo")  # last 3 months of daily data

        if not hist.empty:
            avg_volume = hist["Volume"].mean()
            volume_data[ticker] = avg_volume

    except Exception as e:
        pass

    if i % 50 == 0:
        print(f"Processed {i}/{len(tickers)}...")

    time.sleep(0.1)

volume_df = pd.DataFrame.from_dict(
    volume_data, orient="index", columns=["avg_daily_volume"]
).reset_index()
volume_df.columns = ["ticker", "avg_daily_volume"]

print(f"\n Volume data pulled for {len(volume_df)} stocks")
print(f"\nSample:")
print(volume_df.sort_values("avg_daily_volume", ascending=False).head(10))

Pulling volume data for 188 stocks...
Processed 0/188...


ERROR:yfinance:$UTH: possibly delisted; no price data found  (period=3mo)
ERROR:yfinance:Failed to get ticker 'WDC 3 11/15/28' reason: unexpected character: line 1 column 1 (char 0)
ERROR:yfinance:HTTP Error 500: <!DOCTYPE html>
<html lang="en-us">
  <head>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta charset="utf-8">
    <title>Yahoo</title>
    <meta name="viewport" content="width=device-width,initial-scale=1,minimal-ui">
    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
    <style>
      html {
          height: 100%;
      }
      body {
          background: #fafafc url(https://s.yimg.com/nn/img/sad-panda-201402200631.png) 50% 50%;
          background-size: cover;
          height: 100%;
          text-align: center;
          font: 300 18px "helvetica neue", helvetica, verdana, tahoma, arial, sans-serif;
          margin: 0;
      }
      table {
          height: 100%;
          width: 100%;
          table-layout: fixed;
  

Processed 50/188...


ERROR:yfinance:Failed to get ticker 'BABA 0.5 06/01/31' reason: unexpected character: line 1 column 1 (char 0)
ERROR:yfinance:HTTP Error 500: <!DOCTYPE html>
<html lang="en-us">
  <head>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta charset="utf-8">
    <title>Yahoo</title>
    <meta name="viewport" content="width=device-width,initial-scale=1,minimal-ui">
    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
    <style>
      html {
          height: 100%;
      }
      body {
          background: #fafafc url(https://s.yimg.com/nn/img/sad-panda-201402200631.png) 50% 50%;
          background-size: cover;
          height: 100%;
          text-align: center;
          font: 300 18px "helvetica neue", helvetica, verdana, tahoma, arial, sans-serif;
          margin: 0;
      }
      table {
          height: 100%;
          width: 100%;
          table-layout: fixed;
          border-collapse: collapse;
          border-spacing: 0;
       

Processed 100/188...


ERROR:yfinance:$6RJ0: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TRVC: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:Failed to get ticker 'LITE 1.5 12/15/29' reason: unexpected character: line 1 column 1 (char 0)
ERROR:yfinance:$LITE 1.5 12/15/29: possibly delisted; no price data found  (period=3mo)
ERROR:yfinance:$MIGA: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")


Processed 150/188...


ERROR:yfinance:$DWD: possibly delisted; no price data found  (period=3mo)
ERROR:yfinance:$T7D: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$DYH: possibly delisted; no price data found  (period=3mo)



 Volume data pulled for 172 stocks

Sample:
    ticker  avg_daily_volume
0     NVDA      1.739234e+08
59    INTC      1.026090e+08
1      SPY      7.960977e+07
8     TSLA      6.576946e+07
18    IBIT      6.506747e+07
67       F      6.068248e+07
133   SOFI      5.672633e+07
161    AAL      5.605382e+07
25    NFLX      5.135263e+07
23    AAPL      4.795303e+07


In [ ]:
# Add ticker column to holdings using our CUSIP mapping
holdings_df["ticker"] = holdings_df["cusip"].map(cusip_ticker_map)

# Merge holdings with volume data
merged_df = holdings_df.merge(volume_df, on="ticker", how="inner")

# Sanity check
print(f" Merged dataset: {len(merged_df)} rows")
print(f" Funds: {merged_df['fund'].nunique()}")
print(f" Unique stocks: {merged_df['ticker'].nunique()}")
print(f"\nSample:")
print(merged_df[["fund","stock","ticker","shares","value_usd","avg_daily_volume"]].head())

 Merged dataset: 1223 rows
 Funds: 8
 Unique stocks: 172

Sample:
                    fund                       stock ticker   shares  \
0  Two Sigma Investments      ABERCROMBIE & FITCH CO    ANF    79484   
1  Two Sigma Investments                   ADOBE INC   ADBE   741976   
2  Two Sigma Investments  ADVANCED MICRO DEVICES INC    AMD  1342291   
3  Two Sigma Investments                  AIRBNB INC   ABNB  1913495   
4  Two Sigma Investments     AKAMAI TECHNOLOGIES INC   AKAM  2048557   

   value_usd  avg_daily_volume  
0   10004651      1.743993e+06  
1  259684180      5.122737e+06  
2  287465041      3.561959e+07  
3  259699541      4.805955e+06  
4  178736598      3.558759e+06  


Calculating crowding score per stock.

1. Days ADV = total shares held by ALL funds / average daily volume (Interpretation: how many days would it take all funds to exit this position)

In [ ]:
crowding_df = (
    merged_df.groupby(["ticker", "stock"])
    .agg(
        total_shares_held = ("shares", "sum"),
        total_value_usd   = ("value_usd", "sum"),
        num_funds_holding = ("fund", "nunique"),
        avg_daily_volume  = ("avg_daily_volume", "first")
    )
    .reset_index()
)

# The core crowding metric
crowding_df["days_adv"] = (
    crowding_df["total_shares_held"] / crowding_df["avg_daily_volume"]
)

# Normalize to a 0 to 100 crowding score
crowding_df["crowding_score"] = (
    (crowding_df["days_adv"] - crowding_df["days_adv"].min()) /
    (crowding_df["days_adv"].max() - crowding_df["days_adv"].min()) * 100
)


In [ ]:
# Sort by most crowded
crowding_df = crowding_df.sort_values("crowding_score", ascending=False)

print(f" Crowding scores calculated for {len(crowding_df)} stocks")
print(f"\nTop 15 most crowded stocks right now:")
print(
    crowding_df[["stock","ticker","days_adv","crowding_score","num_funds_holding"]]
    .head(15)
    .to_string(index=False)
)

 Crowding scores calculated for 172 stocks

Top 15 most crowded stocks right now:
                       stock ticker   days_adv  crowding_score  num_funds_holding
        GOLDMAN SACHS ETF TR   GSID 513.912026      100.000000                  2
        GOLDMAN SACHS ETF TR   GSUS 147.216509       28.640443                  2
BRIGHT HORIZONS FAM SOL IN D   BFAM   6.532552        1.263111                  8
           MEDPACE HLDGS INC   MEDP   6.282459        1.214443                  7
                VERISIGN INC   VRSN   5.570017        1.075800                  8
           SPDR SERIES TRUST   SPYV   5.163685        0.996727                  4
                 ISHARES INC    EZU   5.038421        0.972351                  3
         PAYLOCITY HLDG CORP   PCTY   4.730039        0.912339                  8
             FRANCO NEV CORP    FNV   3.916403        0.754004                  7
        VANGUARD MALVERN FDS   VTIP   3.753964        0.722394                  2
  NEUROCRINE BIO

Once we had our crowding scores calculated, the next challenge was figuring out which stocks were genuinely dangerous versus just slightly more held than average. Simply ranking by Days ADV was not enough because almost every stock had some level of fund ownership. We needed a smarter way to flag the truly anomalous ones.
This is where Isolation Forest comes in. It looks at three things simultaneously for each stock — how hard it is to exit, how many funds are holding it, and how much total money is concentrated in it. Instead of using a fixed threshold like "flag anything above 5 days ADV", it learns the normal pattern across all 172 stocks and automatically identifies the ones that look dangerously different from that pattern.
The reason this matters is that a stock held by 8 funds with 5 days ADV is far more dangerous than a stock held by 2 funds with 6 days ADV. Isolation Forest captures that combined signal in a way a simple ranking never could.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Features for anomaly detection
# We use three signals together:
# 1. days_adv — how hard to exit
# 2. num_funds_holding — how many funds are crowded in
# 3. total_value_usd — how much money is concentrated here

features = crowding_df[["days_adv", "num_funds_holding", "total_value_usd"]].copy()

# Remove the Goldman ETF outliers before scaling
# They distort the model — we flag them separately
features = features[crowding_df["days_adv"] < 50]
crowding_clean = crowding_df[crowding_df["days_adv"] < 50].copy()

# Scale features — Isolation Forest works better on scaled data
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Train Isolation Forest
# contamination = 0.1 means we expect ~10% of stocks to be anomalous
iso_forest = IsolationForest(
    contamination=0.1,
    random_state=42,
    n_estimators=100
)

crowding_clean["anomaly"] = iso_forest.fit_predict(features_scaled)
crowding_clean["anomaly_score"] = iso_forest.decision_function(features_scaled)

# Flag crowded anomalies
# anomaly = -1 means flagged as outlier by Isolation Forest
crowding_clean["is_crowded"] = crowding_clean["anomaly"] == -1

print(f" Isolation Forest complete")
print(f" Stocks flagged as anomalously crowded: {crowding_clean['is_crowded'].sum()}")
print(f"\nTop crowded anomalies flagged by ML:")
print(
    crowding_clean[crowding_clean["is_crowded"]]
    [["stock", "ticker", "days_adv", "num_funds_holding", "crowding_score", "anomaly_score"]]
    .sort_values("anomaly_score")
    .head(15)
    .to_string(index=False)
)

 Isolation Forest complete
 Stocks flagged as anomalously crowded: 17

Top crowded anomalies flagged by ML:
                       stock ticker  days_adv  num_funds_holding  crowding_score  anomaly_score
          NVIDIA CORPORATION   NVDA  0.307559                  8        0.051717      -0.179831
        VANGUARD MALVERN FDS   VTIP  3.753964                  2        0.722394      -0.173970
         SPDR S&P 500 ETF TR    SPY  0.172512                  8        0.025437      -0.168150
           MEDPACE HLDGS INC   MEDP  6.282459                  7        1.214443      -0.143250
BRIGHT HORIZONS FAM SOL IN D   BFAM  6.532552                  8        1.263111      -0.130478
                 ISHARES INC    EZU  5.038421                  3        0.972351      -0.128394
           SPDR SERIES TRUST   SPYV  5.163685                  4        0.996727      -0.128268
                  ISHARES TR    IVV  0.624437                  7        0.113382      -0.080368
                VERISIGN INC

The Isolation Forest flagged 17 stocks out of 172 as anomalously crowded. Two clear risk patterns emerged.

The first is universal ownership — stocks like BRIGHT HORIZONS, VERISIGN and PAYLOCITY are held by all 8 funds simultaneously, meaning a single sell decision could trigger a chain reaction.

The second is exit difficulty — BRIGHT HORIZONS and VERISIGN would take over 5 days to exit under normal trading conditions, making them the most dangerous positions in our dataset.

Most unexpectedly, Ethereum ETFs appeared in the flagged list, suggesting quant funds are quietly building crypto exposure through a largely unmonitored channel.

Build the Visualizations

Top 20 Most Crowded Stocks (Bar Chart)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Chart 1 — Top 20 Most Crowded Stocks (Bar Chart) ──────────────────

top20 = crowding_clean.head(20)

fig1 = px.bar(
    top20,
    x="ticker",
    y="crowding_score",
    color="num_funds_holding",
    color_continuous_scale="Reds",
    title="Top 20 Most Crowded Stocks Across Major Quant Funds",
    labels={
        "ticker": "Stock",
        "crowding_score": "Crowding Score",
        "num_funds_holding": "Number of Funds Holding"
    },
    hover_data=["stock", "days_adv", "num_funds_holding"]
)

fig1.update_layout(
    plot_bgcolor="#0e0c09",
    paper_bgcolor="#0e0c09",
    font_color="white",
    title_font_size=16
)

fig1.show()

Crowding vs Exit Difficulty Scatter Plot

In [ ]:
# ── Chart 2 — Crowding Score vs Days ADV Scatter ──────────────────────

fig2 = px.scatter(
    crowding_clean,
    x="days_adv",
    y="num_funds_holding",
    size="total_value_usd",
    color="is_crowded",
    color_discrete_map={True: "#ff4444", False: "#4444ff"},
    hover_data=["stock", "ticker", "crowding_score", "days_adv"],
    title="Crowding Risk Map — Exit Difficulty vs Number of Funds Holding",
    labels={
        "days_adv": "Days to Exit (Days ADV)",
        "num_funds_holding": "Number of Funds Holding",
        "is_crowded": "Flagged as Crowded",
        "total_value_usd": "Total Value USD"
    }
)

# Add quadrant labels
fig2.add_annotation(
    x=5, y=7.5,
    text="DANGER ZONE",
    showarrow=False,
    font=dict(color="#ff4444", size=14),
)

fig2.add_annotation(
    x=0.5, y=2,
    text="LOW RISK",
    showarrow=False,
    font=dict(color="#44ff44", size=14),
)

fig2.update_layout(
    plot_bgcolor="#0e0c09",
    paper_bgcolor="#0e0c09",
    font_color="white",
    title_font_size=16
)

fig2.show()

 Fund Overlap Heatmap

In [ ]:
# ── Chart 3 — Fund Overlap Heatmap ────────────────────────────────────

# Build overlap matrix — how many stocks do any two funds share
funds_list = merged_df["fund"].unique()
overlap_matrix = pd.DataFrame(index=funds_list, columns=funds_list, data=0)

for fund1 in funds_list:
    for fund2 in funds_list:
        stocks1 = set(merged_df[merged_df["fund"]==fund1]["ticker"])
        stocks2 = set(merged_df[merged_df["fund"]==fund2]["ticker"])
        overlap = len(stocks1.intersection(stocks2))
        overlap_matrix.loc[fund1, fund2] = overlap

overlap_matrix = overlap_matrix.astype(int)

fig3 = px.imshow(
    overlap_matrix,
    title="Fund Portfolio Overlap — How Many Stocks Do Funds Share?",
    color_continuous_scale="Reds",
    labels=dict(color="Shared Stocks")
)

fig3.update_layout(
    plot_bgcolor="#0e0c09",
    paper_bgcolor="#0e0c09",
    font_color="white",
    title_font_size=16
)

fig3.show()

Anomaly Score Distribution

In [ ]:
# ── Chart 4 — Isolation Forest Anomaly Scores ─────────────────────────

fig4 = px.scatter(
    crowding_clean,
    x="ticker",
    y="anomaly_score",
    color="is_crowded",
    color_discrete_map={True: "#ff4444", False: "#aaaaaa"},
    hover_data=["stock", "num_funds_holding", "days_adv"],
    title="Isolation Forest Anomaly Scores — Flagged Stocks in Red",
    labels={
        "ticker": "Stock",
        "anomaly_score": "Anomaly Score (lower = more anomalous)",
        "is_crowded": "Flagged as Crowded"
    }
)

fig4.add_hline(
    y=0,
    line_dash="dash",
    line_color="yellow",
    annotation_text="Anomaly Threshold",
    annotation_position="top right"
)

fig4.update_layout(
    plot_bgcolor="#0e0c09",
    paper_bgcolor="#0e0c09",
    font_color="white",
    title_font_size=16
)

fig4.show()

**Dataset 3 — Ken French Factor Returns**

In [ ]:
import requests
import zipfile
import io

# Download Fama French 5 Factor data directly
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"

print("Downloading Ken French Factor data...")
r = requests.get(url)

# Unzip in memory
z = zipfile.ZipFile(io.BytesIO(r.content))
filename = z.namelist()[0]
print(f"File inside zip: {filename}")

# Read the CSV — skip first few rows which are headers/notes
with z.open(filename) as f:
    raw = f.read().decode("utf-8")

# Find where actual data starts
lines = raw.split("\n")
start_line = 0
for i, line in enumerate(lines):
    if line.strip().startswith("2"):  # data starts with year like 20140101
        start_line = i
        break

print(f"Data starts at line: {start_line}")
print(f"Sample line: {lines[start_line]}")

File inside zip: F-F_Research_Data_5_Factors_2x3_daily.csv
Data starts at line: 9197
Sample line: 20000103,   -0.71,   -0.09,   -1.31,   -1.48,   -0.70,    0.02


In [ ]:
# Parse the factor data properly
factor_df = pd.read_csv(
    io.StringIO("\n".join(lines[start_line:])),
    header=None,
    names=["date", "mkt_rf", "smb", "hml", "rmw", "cma", "rf"],
    skipinitialspace=True
)

# Remove any non-numeric rows at the end
factor_df = factor_df[pd.to_numeric(factor_df["date"], errors="coerce").notna()]

# Convert date to datetime
factor_df["date"] = pd.to_datetime(factor_df["date"], format="%Y%m%d")

# Convert all factors to numeric
for col in ["mkt_rf", "smb", "hml", "rmw", "cma", "rf"]:
    factor_df[col] = pd.to_numeric(factor_df[col], errors="coerce")

# Filter to last 5 years only
factor_df = factor_df[factor_df["date"] >= "2020-01-01"]
factor_df = factor_df.dropna()
factor_df = factor_df.reset_index(drop=True)

print(f" Factor data loaded: {len(factor_df)} daily observations")
print(f" Date range: {factor_df['date'].min()} to {factor_df['date'].max()}")
print(f"\nSample:")
print(factor_df.head())
print(f"\nColumn meanings:")
print("mkt_rf = Market return above risk free rate")
print("smb    = Small minus Big (size factor)")
print("hml    = High minus Low (value factor)")
print("rmw    = Robust minus Weak (profitability factor)")
print("cma    = Conservative minus Aggressive (investment factor)")

 Factor data loaded: 1528 daily observations
 Date range: 2020-01-02 00:00:00 to 2026-01-30 00:00:00

Sample:
        date  mkt_rf   smb   hml   rmw   cma    rf
0 2020-01-02    0.86 -0.97 -0.34  0.23 -0.22  0.01
1 2020-01-03   -0.67  0.29  0.01 -0.14 -0.17  0.01
2 2020-01-06    0.36 -0.22 -0.55 -0.16 -0.31  0.01
3 2020-01-07   -0.19 -0.04 -0.26 -0.13 -0.31  0.01
4 2020-01-08    0.47 -0.18 -0.64 -0.21 -0.15  0.01

Column meanings:
mkt_rf = Market return above risk free rate
smb    = Small minus Big (size factor)
hml    = High minus Low (value factor)
rmw    = Robust minus Weak (profitability factor)
cma    = Conservative minus Aggressive (investment factor)


In [ ]:
# ── Chart 5 — Factor Performance with Quant Unwind Events ─────────────

# Calculate cumulative returns for each factor
factor_df["mkt_cumulative"] = (1 + factor_df["mkt_rf"]/100).cumprod()
factor_df["hml_cumulative"] = (1 + factor_df["hml"]/100).cumprod()
factor_df["smb_cumulative"] = (1 + factor_df["smb"]/100).cumprod()
factor_df["rmw_cumulative"] = (1 + factor_df["rmw"]/100).cumprod()

fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=factor_df["date"], y=factor_df["mkt_cumulative"],
    name="Market", line=dict(color="#4a90d9", width=2)
))
fig5.add_trace(go.Scatter(
    x=factor_df["date"], y=factor_df["hml_cumulative"],
    name="Value (HML)", line=dict(color="#c8a96e", width=2)
))
fig5.add_trace(go.Scatter(
    x=factor_df["date"], y=factor_df["smb_cumulative"],
    name="Size (SMB)", line=dict(color="#9b6ec8", width=2)
))
fig5.add_trace(go.Scatter(
    x=factor_df["date"], y=factor_df["rmw_cumulative"],
    name="Profitability (RMW)", line=dict(color="#6ec8a0", width=2)
))

# Mark known quant unwind events — using shapes instead of vline
unwind_events = {
    "COVID Crash Mar 2020":     "2020-03-16",
    "Rate Hike Shock Jan 2022": "2022-01-24",
    "Quant Unwind Aug 2025":    "2025-08-05",
}

shapes = []
annotations = []

for label, date in unwind_events.items():
    shapes.append(dict(
        type="line",
        x0=date, x1=date,
        y0=0, y1=1,
        xref="x", yref="paper",
        line=dict(color="#ff4444", width=1.5, dash="dash")
    ))
    annotations.append(dict(
        x=date, y=1,
        xref="x", yref="paper",
        text=label,
        showarrow=False,
        font=dict(color="#ff4444", size=9),
        textangle=-90,
        xanchor="left"
    ))

fig5.update_layout(
    title="Factor Performance 2020 to 2025 — Major Quant Unwind Events Marked",
    xaxis_title="Date",
    yaxis_title="Cumulative Return",
    plot_bgcolor="#0e0c09",
    paper_bgcolor="#0e0c09",
    font_color="white",
    title_font_size=16,
    shapes=shapes,
    annotations=annotations,
    legend=dict(bgcolor="#1a1a2e", bordercolor="#4a90d9")
)

fig5.show()

Dataset 4 — arXiv Research Papers


In [ ]:
import os
import requests
import time

# Create a folder to store papers
os.makedirs("papers", exist_ok=True)

# All 15 papers — direct PDF download links
papers = {
    "crowding_factor_returns.pdf":      "https://arxiv.org/pdf/2011.06743",
    "momentum_crash.pdf":               "https://arxiv.org/pdf/1407.4497",
    "crowding_equity_strategies.pdf":   "https://arxiv.org/pdf/1807.02556",
    "factor_investing_broken.pdf":      "https://arxiv.org/pdf/2012.05840",
    "volatility_quant_signal.pdf":      "https://arxiv.org/pdf/1903.05697",
    "market_impact_quant.pdf":          "https://arxiv.org/pdf/cond-mat/0307350",
    "predicting_returns_alt_data.pdf":  "https://arxiv.org/pdf/2009.01579",
    "big_data_finance.pdf":             "https://arxiv.org/pdf/1904.10912",
    "ml_asset_management.pdf":          "https://arxiv.org/pdf/1811.04564",
    # New additions — all confirmed on arXiv
    "crowding_networks.pdf":            "https://arxiv.org/pdf/2306.08105",
    "quant_factor_engineering.pdf":     "https://arxiv.org/pdf/2507.07107",
    "systemic_risk_analytics.pdf":      "https://www.financialresearch.gov/working-papers/files/OFRwp0001_BisiasFloodLoValavanis_ASurveyOfSystemicRiskAnalytics.pdf",
    "deep_learning_finance.pdf":        "https://arxiv.org/pdf/2012.04668",
    "factor_momentum.pdf":              "https://arxiv.org/pdf/1907.10003",
    "liquidity_asset_pricing.pdf":      "https://arxiv.org/pdf/2005.02239",
}

# SSRN papers need manual download — we will handle separately
# For now download the 9 arXiv papers automatically

headers_pdf = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Downloading research papers...")
downloaded = []
failed = []

for filename, url in papers.items():
    try:
        r = requests.get(url, headers=headers_pdf, timeout=30)
        if r.status_code == 200 and len(r.content) > 10000:
            with open(f"papers/{filename}", "wb") as f:
                f.write(r.content)
            print(f" {filename}")
            downloaded.append(filename)
        else:
            print(f" {filename} — status {r.status_code}")
            failed.append(filename)
    except Exception as e:
        print(f" {filename} — {e}")
        failed.append(filename)
    time.sleep(2)

print(f"\n Downloaded: {len(downloaded)} papers")
print(f" Failed: {len(failed)} papers")

 crowding_factor_returns.pdf
 momentum_crash.pdf
 crowding_equity_strategies.pdf
 factor_investing_broken.pdf
 volatility_quant_signal.pdf
 market_impact_quant.pdf
 predicting_returns_alt_data.pdf
 big_data_finance.pdf
 ml_asset_management.pdf
 crowding_networks.pdf
 quant_factor_engineering.pdf
 systemic_risk_analytics.pdf
 deep_learning_finance.pdf
 factor_momentum.pdf
 liquidity_asset_pricing.pdf

 Downloaded: 15 papers
 Failed: 0 papers


In [ ]:
!pip install langchain langchain-groq chromadb sentence-transformers pypdf -q
print(" RAG libraries installed")

 RAG libraries installed


In [ ]:
!pip install langchain-text-splitters -q

Building the RAG Knowledge Base

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

print("Loading and chunking papers...")

all_docs = []
for filename in os.listdir("papers"):
    if filename.endswith(".pdf"):
        try:
            loader = PyPDFLoader(f"papers/{filename}")
            docs = loader.load()
            for doc in docs:
                doc.metadata["source"] = filename
            all_docs.extend(docs)
            print(f" Loaded: {filename} — {len(docs)} pages")
        except Exception as e:
            print(f" {filename} — {e}")

print(f"\n Total pages loaded: {len(all_docs)}")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)
print(f" Total chunks created: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[10].page_content[:300])

Loading and chunking papers...
 Loaded: quant_factor_engineering.pdf — 9 pages
 Loaded: market_impact_quant.pdf — 4 pages


 Loaded: crowding_factor_returns.pdf — 10 pages
 Loaded: crowding_networks.pdf — 8 pages
 Loaded: crowding_equity_strategies.pdf — 29 pages


 Loaded: predicting_returns_alt_data.pdf — 16 pages
 Loaded: factor_investing_broken.pdf — 17 pages
 Loaded: momentum_crash.pdf — 8 pages
 Loaded: liquidity_asset_pricing.pdf — 4 pages
 Loaded: deep_learning_finance.pdf — 15 pages
 Loaded: big_data_finance.pdf — 16 pages
 Loaded: ml_asset_management.pdf — 6 pages
 Loaded: systemic_risk_analytics.pdf — 165 pages
 Loaded: factor_momentum.pdf — 34 pages
 Loaded: volatility_quant_signal.pdf — 7 pages

 Total pages loaded: 348
 Total chunks created: 2488

Sample chunk:
A. Factor Models and Engineering
Factor-based investing traces its foundations to the Capi-
tal Asset Pricing Model (CAPM) [1] and Arbitrage Pricing
Theory (APT) [2], establishing the mathematical framework
for multi-factor return decomposition. The Fama-French th ree-
factor and ﬁve-factor models i


Building the vector database

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Building vector database...")
#print("This will take 3 to 5 minutes — embedding 2488 chunks...")

# Use a small but effective embedding model
# all-MiniLM-L6-v2 is fast, free, and works well for financial text
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f" Vector database built successfully")
print(f" Total chunks embedded: {vectorstore._collection.count()}")
print(f"\nTesting retrieval...")

# Quick test — search for crowding related content
test_results = vectorstore.similarity_search(
    "what causes quant fund crowding and unwind events",
    k=3
)

print(f" Retrieval working — top result preview:")
print(test_results[0].page_content[:300])
print(f"\nSource: {test_results[0].metadata['source']}")

Building vector database...


/tmp/ipykernel_364/3250434485.py:9: LangChainDeprecationWarning:

The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Vector database built successfully
 Total chunks embedded: 2488

Testing retrieval...
 Retrieval working — top result preview:
risk	associated	with	investors’	positioning.	Therefore,	understanding	and	deTining	‘crowding’	is	vital	for	investment	management.		Crowding	 is	 a	term	 widely	used	 in	 industry,	 albeit	 not	 always	 in	the	same	 way	 or	 using	 the	 same	methodology.	Crowding	is	often	mentioned	as	a	single	most	i

Source: crowding_networks.pdf


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

os.environ["GROQ_API_KEY"] = "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"  # paste your key

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",  # updated model name
    temperature=0.1,
    max_tokens=1000
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = PromptTemplate.from_template("""
You are a senior quantitative analyst at a hedge fund with deep expertise in
factor investing, crowding risk, and systematic strategies.

Use the following research paper excerpts to answer the question.
Always cite which paper your answer comes from.
Keep answers concise and accessible to a non-technical audience.

Research excerpts:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(" RAG chain built")
print("\nTesting...")

answer = rag_chain.invoke(
    "What causes quant funds to crowd into the same positions?"
)
print("\nAI Answer:")
print(answer)

 RAG chain built

Testing...

AI Answer:
According to [no specific reference provided in the excerpts], crowding in quant funds can be caused by various factors, but it's often related to the risk associated with investors' positioning. One approach to understanding crowding is based on analysis at the factor level, rather than a single stock level, and correlation patterns of stock returns [1]. Additionally, holding-based approaches, such as network analysis of fund holdings, can also be used to identify crowding [2].


Test Questions

In [ ]:
test_questions = [
    "What happened during the August 2007 quant unwind and what were the losses?",
    "How does momentum factor crowding lead to crashes?",
    "What is the relationship between liquidity and crowding risk?"
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    print("=" * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print()


Question: What happened during the August 2007 quant unwind and what were the losses?
During the August 2007 quant unwind, quantitative factor portfolios were being deleveraged and unwound, causing price impacts to substantially rise starting on August 10 and remain high for the following week (Khandani and Lo, 2007; Khandani and Lo, 2011). The losses were significant, with total expected losses peaking at 1% of GDP in March 2009, and over 50% of these losses could have been transferred to the government at the peak of the crisis (Khandani, Lo, and Merton, 2009).


Question: How does momentum factor crowding lead to crashes?
According to the research excerpts, momentum factor crowding can lead to undesirable features such as negative mean return and negative skewness (Zlotnikov et. al, 2022 [6]). Specifically, the least crowded quintile has constant negative exposure to momentum, suggesting that crowded momentum stocks may be more prone to crashes due to their negative skewness and lo

In [ ]:
import pickle

# Save everything Streamlit needs
with open("crowding_data.pkl", "wb") as f:
    pickle.dump({
        "crowding_clean": crowding_clean,
        "merged_df":      merged_df,
        "factor_df":      factor_df,
        "holdings_df":    holdings_df
    }, f)

print(" Data saved to crowding_data.pkl")
print(" ChromaDB already persisted to ./chroma_db")
print("\nReady to build Streamlit dashboard")

 Data saved to crowding_data.pkl
 ChromaDB already persisted to ./chroma_db

Ready to build Streamlit dashboard


**Building Streamlit Dashboard**

In [ ]:
!pip install pyngrok -q


In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 29.4 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok

# Paste your token here
ngrok.set_auth_token("37s2Beu74jwq2t1OIEi66oHmStd_7qPF6J5V8yqZyAjBt4xoz")

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pickle
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

st.set_page_config(
    page_title="Quant Crowding Detector",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url("https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@300;400;500&family=IBM+Plex+Sans:wght@300;400;600&display=swap");
html, body, [class*="css"] {
    font-family: IBM Plex Sans, sans-serif;
    background-color: #0a0a0f;
    color: #c8d0e0;
}
.stApp { background-color: #0a0a0f; }
[data-testid="stSidebar"] {
    background-color: #0d0d14;
    border-right: 1px solid #1a1a2e;
}
.metric-container {
    background: #111118;
    border: 1px solid #1a1a2e;
    border-left: 3px solid #c8a96e;
    padding: 12px 16px;
    margin-bottom: 10px;
}
.metric-value {
    font-family: IBM Plex Mono, monospace;
    font-size: 28px;
    font-weight: 500;
    color: #e8f0ff;
    line-height: 1;
}
.metric-label {
    font-size: 10px;
    color: #4a5070;
    margin-top: 4px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.metric-danger { border-left-color: #ff4444 !important; }
.metric-warning { border-left-color: #ffaa00 !important; }
.metric-good   { border-left-color: #4caf82 !important; }
.section-label {
    font-family: IBM Plex Mono, monospace;
    font-size: 9px;
    letter-spacing: 0.3em;
    color: #3a4060;
    text-transform: uppercase;
    padding-bottom: 6px;
    border-bottom: 1px solid #1a1a2e;
    margin-bottom: 12px;
    margin-top: 20px;
}
.chat-response {
    background: #111118;
    border-left: 3px solid #c8a96e;
    padding: 16px;
    margin-top: 12px;
}
.insight-box {
    background: #0d1a0d;
    border: 1px solid #1a3a1a;
    border-left: 3px solid #4caf82;
    padding: 16px;
    margin-bottom: 16px;
}
.source-tag {
    font-family: IBM Plex Mono, monospace;
    font-size: 9px;
    color: #3a4060;
    margin-top: 8px;
}
.live-dot {
    width: 8px; height: 8px;
    border-radius: 50%;
    background: #ff4444;
    display: inline-block;
}
</style>
""", unsafe_allow_html=True)

@st.cache_data
def load_data():
    with open("crowding_data.pkl", "rb") as f:
        data = pickle.load(f)
    return (
        data["crowding_clean"],
        data["merged_df"],
        data["factor_df"],
        data["holdings_df"]
    )

@st.cache_resource
def load_rag():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings
    )
    llm = ChatGroq(
        model_name="llama-3.3-70b-versatile",
        temperature=0.1,
        max_tokens=1000,
        api_key=os.environ.get("GROQ_API_KEY", "")
    )
    prompt = PromptTemplate.from_template(
        "You are a senior quantitative analyst at a hedge fund with deep expertise in "
        "factor investing, crowding risk, and systematic strategies. "
        "You have access to BOTH live portfolio data from SEC 13F filings AND "
        "academic research papers on quant crowding. "
        "Use both sources to answer. Always cite the research paper you reference. "
        "When live data is provided, use specific numbers from it in your answer. "
        "Keep answers concise and accessible.\\n\\n"
        "Research paper excerpts:\\n{context}\\n\\n"
        "Question (may include live data):\\n{question}\\n\\n"
        "Answer:"
    )
    def format_docs(docs):
        return "\\n\\n".join(doc.page_content for doc in docs)
    chain = (
        {"context": vectorstore.as_retriever(search_kwargs={"k": 4}) | format_docs,
         "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, vectorstore

crowding_clean, merged_df, factor_df, holdings_df = load_data()
rag_chain, vectorstore = load_rag()

def build_data_context(crowding_clean, merged_df):
    top_flagged = (
        crowding_clean[crowding_clean["is_crowded"]]
        [["stock","ticker","days_adv","num_funds_holding","crowding_score","total_value_usd"]]
        .head(10)
        .to_string(index=False)
    )
    fund_summary = merged_df.groupby("fund").agg(
        stocks_held=("ticker","nunique"),
        total_value=("value_usd","sum")
    ).reset_index().to_string(index=False)

    most_dangerous = (
        crowding_clean[crowding_clean["is_crowded"]]
        .sort_values("days_adv", ascending=False)
        .iloc[0]
    )

    context = (
        "=== LIVE DATA FROM SEC 13F FILINGS ===\\n"
        "Top flagged crowded stocks:\\n" + top_flagged +
        "\\n\\nFund summary:\\n" + fund_summary +
        "\\n\\nSingle most dangerous position: "
        + str(most_dangerous["stock"])
        + " (ticker: " + str(most_dangerous["ticker"]) + ")"
        + ", Days ADV: " + str(round(float(most_dangerous["days_adv"]), 2))
        + ", Held by " + str(int(most_dangerous["num_funds_holding"])) + " funds"
        + ", Total value: $" + str(round(float(most_dangerous["total_value_usd"])/1e6, 1)) + "M"
        + "\\n=================================\\n"
    )
    return context

with st.sidebar:
    st.markdown(
        "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:11px;"
        "color:#c8a96e;letter-spacing:0.15em;margin-bottom:4px\\">"
        "● QUANT CROWDING DETECTOR</div>"
        "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:8px;"
        "color:#3a4060;letter-spacing:0.2em;margin-bottom:20px\\">"
        "SEC 13F · ISOLATION FOREST · RAG</div>",
        unsafe_allow_html=True
    )
    st.markdown('<div class="section-label">Overview</div>', unsafe_allow_html=True)
    st.markdown(
        "<div class=\\"metric-container metric-good\\">"
        "<div class=\\"metric-value\\">" + str(merged_df["fund"].nunique()) + "</div>"
        "<div class=\\"metric-label\\">Funds Monitored</div></div>"
        "<div class=\\"metric-container\\">"
        "<div class=\\"metric-value\\">" + str(crowding_clean["ticker"].nunique()) + "</div>"
        "<div class=\\"metric-label\\">Stocks Tracked</div></div>"
        "<div class=\\"metric-container metric-danger\\">"
        "<div class=\\"metric-value\\">" + str(int(crowding_clean["is_crowded"].sum())) + "</div>"
        "<div class=\\"metric-label\\">ML Flagged Anomalies</div></div>"
        "<div class=\\"metric-container metric-warning\\">"
        "<div class=\\"metric-value\\">" + str(round(float(crowding_clean["days_adv"].max()), 1)) + "</div>"
        "<div class=\\"metric-label\\">Max Days ADV</div></div>",
        unsafe_allow_html=True
    )
    st.markdown('<div class="section-label">Filters</div>', unsafe_allow_html=True)
    min_funds      = st.slider("Min Funds Holding", 1, 8, 3)
    anomalies_only = st.toggle("Anomalies Only", value=False)
    min_days       = st.slider("Min Days ADV", 0.0, 7.0, 0.0, 0.1)
    st.markdown('<div class="section-label">Data Sources</div>', unsafe_allow_html=True)
    st.markdown(
        "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:9px;line-height:2\\">"
        "<span style=\\"color:#4caf82\\">✓</span> SEC EDGAR 13F<br>"
        "<span style=\\"color:#4caf82\\">✓</span> yfinance Volume<br>"
        "<span style=\\"color:#4caf82\\">✓</span> French Factors 2025<br>"
        "<span style=\\"color:#4caf82\\">✓</span> 15 Research Papers</div>",
        unsafe_allow_html=True
    )

st.markdown(
    "<div style=\\"display:flex;align-items:center;gap:12px;padding:8px 0 20px;"
    "border-bottom:1px solid #1a1a2e;margin-bottom:20px\\">"
    "<span class=\\"live-dot\\"></span>"
    "<div><span style=\\"font-size:22px;font-weight:300;color:#e8f0ff;"
    "letter-spacing:-0.01em\\">Crowding Risk Monitor</span>"
    "<span style=\\"font-size:11px;color:#4a5070;font-style:italic;margin-left:16px\\">"
    "Isolation Forest · Real SEC Data · RAG-Powered Analysis</span></div></div>",
    unsafe_allow_html=True
)

filtered = crowding_clean[
    (crowding_clean["num_funds_holding"] >= min_funds) &
    (crowding_clean["days_adv"] >= min_days)
]
if anomalies_only:
    filtered = filtered[filtered["is_crowded"]]

col1, col2 = st.columns(2)

with col1:
    st.markdown('<div class="section-label">Top 20 Most Crowded Stocks</div>',
                unsafe_allow_html=True)
    fig1 = px.bar(
        filtered.head(20), x="ticker", y="crowding_score",
        color="num_funds_holding",
        color_continuous_scale="Reds",
        hover_data=["stock","days_adv","num_funds_holding"],
        labels={"crowding_score":"Crowding Score","ticker":"Stock",
                "num_funds_holding":"Funds Holding"}
    )
    fig1.update_layout(
        plot_bgcolor="#0d0d14", paper_bgcolor="#0d0d14",
        font_color="#8a90a8", height=280,
        margin=dict(l=0,r=0,t=10,b=0)
    )
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    st.markdown('<div class="section-label">Crowding Risk Map</div>',
                unsafe_allow_html=True)
    fig2 = px.scatter(
        filtered, x="days_adv", y="num_funds_holding",
        size="total_value_usd",
        color="is_crowded",
        color_discrete_map={True:"#ff4444", False:"#4a6090"},
        hover_data=["stock","ticker","crowding_score"],
        labels={"days_adv":"Days to Exit (ADV)",
                "num_funds_holding":"Funds Holding",
                "is_crowded":"ML Flagged"}
    )
    fig2.add_annotation(
        x=filtered["days_adv"].max()*0.75,
        y=filtered["num_funds_holding"].max(),
        text="DANGER ZONE", showarrow=False,
        font=dict(color="#ff4444", size=11)
    )
    fig2.update_layout(
        plot_bgcolor="#0d0d14", paper_bgcolor="#0d0d14",
        font_color="#8a90a8", height=280,
        margin=dict(l=0,r=0,t=10,b=0)
    )
    st.plotly_chart(fig2, use_container_width=True)

st.markdown(
    '<div class="section-label">ML Flagged Stocks — Anomalously Crowded Positions</div>',
    unsafe_allow_html=True
)
flagged = filtered[filtered["is_crowded"]].copy()
flagged["status"] = flagged["days_adv"].apply(
    lambda x: "DANGER" if x > 4 else "WATCH"
)
flagged["total_value_fmt"] = (
    (flagged["total_value_usd"]/1e6).round(1).astype(str) + "M"
)
st.dataframe(
    flagged[["stock","ticker","days_adv","num_funds_holding",
             "crowding_score","total_value_fmt","status"]]
    .rename(columns={
        "stock":"Company","ticker":"Ticker","days_adv":"Days ADV",
        "num_funds_holding":"Funds Holding","crowding_score":"Crowding Score",
        "total_value_fmt":"Total Value","status":"Status"
    }).reset_index(drop=True),
    use_container_width=True, height=250
)

col3, col4 = st.columns(2)

with col3:
    st.markdown('<div class="section-label">Factor Performance 2020 — 2025</div>',
                unsafe_allow_html=True)
    factor_df["mkt_cum"] = (1 + factor_df["mkt_rf"]/100).cumprod()
    factor_df["hml_cum"] = (1 + factor_df["hml"]/100).cumprod()
    factor_df["smb_cum"] = (1 + factor_df["smb"]/100).cumprod()
    factor_df["rmw_cum"] = (1 + factor_df["rmw"]/100).cumprod()
    fig5 = go.Figure()
    for col_name, label, color in [
        ("mkt_cum","Market","#4a90d9"),
        ("hml_cum","Value HML","#c8a96e"),
        ("smb_cum","Size SMB","#9b6ec8"),
        ("rmw_cum","Profitability","#4caf82")
    ]:
        fig5.add_trace(go.Scatter(
            x=factor_df["date"], y=factor_df[col_name],
            name=label, line=dict(color=color, width=1.5)
        ))
    for date, label in [
        ("2020-03-16","COVID"),
        ("2022-01-24","Rate Hike"),
        ("2025-08-05","2025 Unwind")
    ]:
        fig5.add_shape(type="line", x0=date, x1=date, y0=0, y1=1,
                       xref="x", yref="paper",
                       line=dict(color="#ff4444", width=1, dash="dash"))
        fig5.add_annotation(x=date, y=0.98, xref="x", yref="paper",
                            text=label, showarrow=False,
                            font=dict(color="#ff4444", size=8),
                            textangle=-90, xanchor="left")
    fig5.update_layout(
        plot_bgcolor="#0d0d14", paper_bgcolor="#0d0d14",
        font_color="#8a90a8", height=280,
        margin=dict(l=0,r=0,t=10,b=0),
        legend=dict(bgcolor="#111118", font=dict(size=9))
    )
    st.plotly_chart(fig5, use_container_width=True)

with col4:
    st.markdown('<div class="section-label">Crowded Positions — Treemap</div>',
                unsafe_allow_html=True)
    treemap_df = filtered[filtered["num_funds_holding"] >= 3].copy()
    treemap_df["label"] = (
        treemap_df["ticker"] + " | " +
        treemap_df["num_funds_holding"].astype(str) + " funds"
    )
    fig7 = px.treemap(
        treemap_df,
        path=["label"],
        values="total_value_usd",
        color="num_funds_holding",
        color_continuous_scale="Reds",
        hover_data=["stock","days_adv","crowding_score"],
        labels={"num_funds_holding":"Funds Holding",
                "total_value_usd":"Total Value USD"}
    )
    fig7.update_traces(
        textfont=dict(family="IBM Plex Mono", size=11, color="white"),
        marker=dict(cornerradius=2)
    )
    fig7.update_layout(
        plot_bgcolor="#0d0d14", paper_bgcolor="#0d0d14",
        font_color="#8a90a8", height=280,
        margin=dict(l=0,r=0,t=10,b=0)
    )
    st.plotly_chart(fig7, use_container_width=True)

st.markdown(
    '<div class="section-label">AI Analyst — Market Insight and Q&A</div>',
    unsafe_allow_html=True
)

if "ai_insight" not in st.session_state:
    with st.spinner("Generating AI market insight from live data..."):
        try:
            data_context = build_data_context(crowding_clean, merged_df)
            auto_prompt = (
                data_context
                + "Based on the live data above and your knowledge of quant crowding research, "
                + "write a 3 to 4 sentence analyst insight summary. Mention the most dangerous "
                + "stock by name with its specific Days ADV number. Cite one research paper. "
                + "Sound like a senior quant analyst writing a Monday morning risk brief. "
                + "Do not use bullet points."
            )
            insight = rag_chain.invoke(auto_prompt)
            st.session_state["ai_insight"] = insight
        except Exception as e:
            st.session_state["ai_insight"] = "Insight generation unavailable — " + str(e)

st.markdown(
    "<div class=\\"insight-box\\">"
    "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:8px;"
    "color:#4caf82;letter-spacing:0.2em;margin-bottom:8px\\">"
    "AUTO-GENERATED MARKET INSIGHT</div>"
    "<div style=\\"font-size:13px;color:#a0b0c8;line-height:1.75\\">"
    + st.session_state["ai_insight"] +
    "</div></div>",
    unsafe_allow_html=True
)

st.markdown(
    "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:9px;"
    "color:#3a4060;margin-bottom:16px\\">"
    "Powered by 15 research papers · Llama 3.3 70B · ChromaDB RAG · Live SEC Data"
    "</div>", unsafe_allow_html=True
)

question = st.text_input(
    "",
    placeholder="Ask anything about quant crowding risk...",
    key="question_input"
)

if st.button("Ask", type="primary") and question:
    with st.spinner("Analysing research papers and live crowding data..."):
        try:
            data_context = build_data_context(crowding_clean, merged_df)
            enriched_question = data_context + "User question: " + question
            answer  = rag_chain.invoke(enriched_question)
            docs    = vectorstore.similarity_search(question, k=3)
            sources = list(set(d.metadata.get("source","") for d in docs))
            st.markdown(
                "<div class=\\"chat-response\\">"
                "<div style=\\"font-family:IBM Plex Mono,monospace;font-size:8px;"
                "color:#c8a96e;letter-spacing:0.2em;margin-bottom:8px\\">"
                "AI ANALYST RESPONSE</div>"
                "<div style=\\"font-size:13px;color:#a0b0c8;line-height:1.75\\">"
                + answer +
                "</div>"
                "<div class=\\"source-tag\\">Sources: " + " · ".join(sources) + "</div>"
                "</div>",
                unsafe_allow_html=True
            )
        except Exception as e:
            st.error("Error: " + str(e))
'''

with open("app.py", "w") as f:
    f.write(app_code)

print(" app.py written successfully")

 app.py written successfully


In [ ]:
import subprocess
import threading
import time
import os
from pyngrok import ngrok

# Set Groq API key
os.environ["GROQ_API_KEY"] = "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"  # paste your key

# Kill any existing streamlit processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Launch Streamlit in background
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Wait for streamlit to start
print("Starting Streamlit...")
time.sleep(8)

# Open ngrok tunnel
tunnel = ngrok.connect(8501)
print(f"\n Dashboard is LIVE")
print(f"\n Open this URL in your browser:")
print(f"\n   {tunnel.public_url}\n")
print("Keep this cell running — closing it will shut down the dashboard")

Starting Streamlit...

 Dashboard is LIVE

 Open this URL in your browser:

   https://romelia-vapoury-jaliyah.ngrok-free.dev

Keep this cell running — closing it will shut down the dashboard


**Conclusion**

This project shows that quant crowding risk is detectable using entirely public data. Across 8 major funds and 172 stocks, the Isolation Forest flagged 17 anomalously crowded positions — with BRIGHT HORIZONS and VERISIGN carrying the highest unwind risk. The RAG-powered AI analyst connects live portfolio signals to academic research, creating an early warning system that could have identified the conditions leading into the 2025 quant unwind before it happened
